In [ ]:
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Tải dữ liệu từ các file .pkl
def load_embeddings(file_list):
    embeddings = []
    for file_name in file_list:
        with open(file_name, "rb") as f:
            data = pickle.load(f)  # Load dữ liệu (giả sử là numpy array)
            embeddings.append(data)
    return np.vstack(embeddings)  # Kết hợp tất cả embeddings thành 1 numpy array

# Danh sách file .pkl
file_list = [f"embedding_{i}.pkl" for i in range(10)]
all_embeddings = load_embeddings(file_list)
print(f"Shape of all_embeddings: {all_embeddings.shape}")

ModuleNotFoundError: No module named 'faiss'

In [ ]:
# Danh sách file .pkl
file_list = [f"D:\\Legal_Document_Retrieval\\data\embeddings_adapted_finetune_phobert\\embeddings_{i}.pkl" for i in range(10)]
all_embeddings = load_embeddings(file_list)
print(f"Shape of all_embeddings: {all_embeddings.shape}")

In [ ]:
# 2. Tạo FAISS index và chuẩn hóa embeddings để sử dụng Cosine Similarity
embedding_size = all_embeddings.shape[1]  # Kích thước vector (ví dụ: 768)

# Chuẩn hóa embeddings (L2 normalization)
def normalize_embeddings(embeddings):
    norm = np.linalg.norm(embeddings, axis=1, keepdims=True)
    return embeddings / norm

all_embeddings_normalized = normalize_embeddings(all_embeddings)

# Tạo FAISS index với cosine similarity
res = faiss.StandardGpuResources()
index_flat = faiss.IndexFlatIP(embedding_size)  # Sử dụng IndexFlatIP cho cosine similarity
index = faiss.index_cpu_to_gpu(res, 0, index_flat)  # Chuyển index lên GPU

# Thêm embeddings đã chuẩn hóa vào FAISS index
index.add(all_embeddings_normalized)
print(f"Số lượng vectors trong index: {index.ntotal}")

# 3. Tải mô hình SentenceTransformer
model = SentenceTransformer('0wovv0/DCT_TRAINST')

# 4. Encode câu query và chuẩn hóa
def encode_query(sentence):
    embedding = model.encode(sentence, convert_to_numpy=True)
    return embedding / np.linalg.norm(embedding)  # Chuẩn hóa vector query

query = "PhoBERT là mô hình mạnh mẽ cho tiếng Việt"
query_vector = encode_query(query).reshape(1, -1)

# 5. Tìm kiếm câu query tương tự
k = 5  # Số lượng kết quả cần tìm
distances, indices = index.search(query_vector, k)
print(f"Indices of nearest neighbors: {indices}")
print(f"Distances to nearest neighbors: {distances}")

# 6. (Tuỳ chọn) Truy xuất câu gốc tương ứng
# Giả sử bạn có danh sách các câu gốc tương ứng với embedding
sentences = ["Câu 1", "Câu 2", "Câu 3", "..."]  # Danh sách câu gốc (phải tương ứng với all_embeddings)
for idx in indices[0]:
    print(f"Nearest neighbor: {sentences[idx]}")

# 7. Lưu FAISS index để tái sử dụng
faiss.write_index(faiss.index_gpu_to_cpu(index), "faiss_index.bin")  # Lưu về CPU trước khi ghi file